# day-28-aws-bedrock — worked solutions & answer key

Solutions to the exercises in [`../lesson.ipynb`](../lesson.ipynb), plus the self-check answer key. **Try each exercise yourself first** — the value is in the attempt, not the answer.

In [8]:
# ---- Solution 1 ----
def to_converse(system, messages):
    sys_blocks = [{"text": system}] if isinstance(system, str) else system
    cm = [{"role": m["role"],
           "content": [{"text": m["content"]}] if isinstance(m["content"], str) else m["content"]}
          for m in messages]
    return sys_blocks, cm

def from_converse(resp):
    msg = resp["output"]["message"]
    text = "".join(b["text"] for b in msg["content"] if "text" in b)
    return dict(role=msg["role"], content=text, stop_reason=resp["stopReason"],
                usage=dict(input_tokens=resp["usage"]["inputTokens"],
                           output_tokens=resp["usage"]["outputTokens"]))

s, m = to_converse("Be terse.", [{"role": "user", "content": "hi"},
                                 {"role": "assistant", "content": "hello"},
                                 {"role": "user", "content": "bye"}])
print("converse messages:", json.dumps(m))
r = brt.converse(modelId="us.anthropic.claude-sonnet-4-5-20250929-v1:0", system=s, messages=m,
                 inferenceConfig={"maxTokens": 100})
print("back to first-party:", from_converse(r))

converse messages: [{"role": "user", "content": [{"text": "hi"}]}, {"role": "assistant", "content": [{"text": "hello"}]}, {"role": "user", "content": [{"text": "bye"}]}]
back to first-party: {'role': 'assistant', 'content': '[bedrock:claude-sonnet-4-5-20] answer to: bye', 'stop_reason': 'end_turn', 'usage': {'input_tokens': 90, 'output_tokens': 11}}


In [9]:
# ---- Solution 3 ----
docs = 2_000_000; in_tok, out_tok = 300, 20
sync = docs * (in_tok*ON_DEMAND["pin"] + out_tok*ON_DEMAND["pout"]) / 1e6
batch = docs * (in_tok*BATCH["pin"] + out_tok*BATCH["pout"]) / 1e6
print(f"S3: sync on-demand ${sync:,.0f}   batch ${batch:,.0f}   save ${sync-batch:,.0f} ({100*(1-batch/sync):.0f}%)")
print("    trade-off: batch results land in S3 minutes-to-hours later, not synchronously.")

S3: sync on-demand $2,400   batch $1,200   save $1,200 (50%)
    trade-off: batch results land in S3 minutes-to-hours later, not synchronously.


### Solutions 2, 4, 5, 6 (sketch)

**S2:** provisioned cost = `ceil(calls/mo /30/24/60 * (in+out) * peak_ratio / UNIT_TPM) *
UNIT_MONTHLY`; on-demand = linear in calls. Break-even calls/mo rises as `peak_ratio` rises
(spikier traffic needs more reserved units for the same average, so on-demand stays cheaper
longer). Plot both vs calls/mo for each ratio; the crossover is the break-even.

**S4:** `apply_guardrail(content, source)` → scan for denied-topic keywords, `re.sub` emails
to `[EMAIL]`, return `{"action": "GUARDRAIL_INTERVENED" | "NONE", "output": redacted}`. Call
it on user input *before* the model (block bad requests cheaply) and on model output *after*
(catch leaks) — independent of the model call, so you can pre-screen a batch or gate a
non-Bedrock model.

**S5:** EU-only → use a **region-pinned foundation model ID** (`anthropic.claude-...-v1:0`) in
an EU region, or the `eu.` inference profile (routes only within EU). You can't use `us.`
because it may route the request to a US region, violating residency. You give up the extra
availability headroom that a wider routing pool provides.

**S6:** (a) **Claude Platform on AWS** or first-party now, Bedrock later — or write against the
Anthropic SDK and switch the client class. (b) **Bedrock** — VPC endpoints, IAM, CloudTrail
are the whole point; accept the feature lag. (c) **First-party Anthropic API** — Bedrock is
weeks-to-months behind on new models; accept managing an API key.

### Answer key
1. Any three: IAM auth instead of API keys; data stays in your AWS region/account; VPC/
   PrivateLink private connectivity; CloudTrail/CloudWatch audit + logging; one API across
   model vendors; provisioned throughput; managed Guardrails/Knowledge Bases/Agents; AWS
   Marketplace billing.
2. Access to the newest models and features on day one — Bedrock lags the first-party API by
   weeks to months, and some features are first-party only.
3. `converse` for a portable, vendor-neutral message API (recommended default); `invoke_model`
   when you need to send a model's exact native request body or use a feature `converse`
   doesn't expose yet.
4. A model ID prefixed `us.` / `eu.` / `apac.` that lets AWS route the request to any region
   in that geo with capacity — higher availability at the same price; use it in production
   unless data residency forces a single pinned region.
5. Start on-demand. Switch to provisioned throughput only when load is steady and high, you're
   being throttled, or the provisioned math beats on-demand at your sustained volume — and
   size for p95, not peak. Use batch for anything asynchronous (~50% off).
6. Knowledge Bases (`retrieve_and_generate`) — point it at S3 and it does chunk/embed/store/
   retrieve/generate/cite. You trade fine control over chunking, hybrid retrieval, and
   reranking for time-to-ship.
7. `bedrock` = control plane (list models, provisioned throughput, guardrails, logging config);
   `bedrock-runtime` = data plane (`converse`, `invoke_model`, `apply_guardrail`).